In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import cv2
import os
from tqdm import tqdm
import pydicom as dicom

In [3]:
def preprocess_image(image, target_size=(224, 224)):
    """Improves image preprocessing with proper normalization and scaling."""
    image = cv2.resize(image, target_size)
    
    image = image.astype('float32')
    image = (image / 255.0)
    image = np.repeat(image[..., np.newaxis], 3, -1)
    
    return image

In [ ]:
def load_data_from_folder(folder_path, lbl, clc, ns):
    images_tr = []
    labels_tr = []
    age = []
    calc = []
    root1 = folder_path + '/Below_40'
    root2 = folder_path + '/Above_40'
    cc =0
    for root, label in [(root1, 0), (root2, 1)]:
        files_list = sorted([f for f in os.listdir(root) if not f.startswith(".")])

        for file in tqdm(files_list):
            try:
                file_path = os.path.join(root, file)
                image_file = dicom.dcmread(file_path)
                image = image_file.pixel_array
                if image is not None:
                    # Apply improved preprocessing
                    image = preprocess_image(image, (ns, ns))
                    images_tr.append(image)
                    labels_tr.append(lbl)
                    age.append(label)
                    calc.append(clc)
            except Exception as e:
                print(f"Error processing {file_path}: {str(e)}")
                continue

    return np.array(images_tr), np.array(labels_tr), np.array(age), np.array(calc)

In [5]:
Benign_path = '/your_own_path/Benign/None'
her2_path1 = '/your_own_path/HER2/mass'
her2_path2 = '/your_own_path/HER2/calcification'
her2_path3 = '/your_own_path/HER2/both'
lumA_path1 = '/your_own_path/LumA/mass'
lumA_path2 = '/your_own_path/LumA/calcification'
lumA_path3 = '/your_own_path/LumA/both'
LumB_path1 = '/your_own_path/LumB/mass'
LumB_path2 = '/your_own_path/LumB/calcification'
LumB_path3 = '/your_own_path/LumB/both'
TN_path1 = '/your_own_path/TN/mass'
TN_path2 = '/your_own_path/TN/calcification'
TN_path3 = '/your_own_path/TN/both'


ns = 224

X_0, y_0, age0, calc0 = load_data_from_folder(Benign_path,0,0, ns)
X_11, y_11, age11, calc11 = load_data_from_folder(her2_path1,1, 1,ns)
X_12, y_12, age12, calc12 = load_data_from_folder(her2_path2,1,2,ns)
X_13, y_13, age13, calc13 = load_data_from_folder(her2_path3,1,3,ns)
X_21, y_21, age21, calc21 = load_data_from_folder(lumA_path1,2,1,ns)
X_22, y_22, age22, calc22 = load_data_from_folder(lumA_path2,2,2,ns)
X_23, y_23, age23, calc23 = load_data_from_folder(lumA_path3,2,3,ns)
X_31, y_31, age31, calc31 = load_data_from_folder(LumB_path1,3,1,ns)
X_32, y_32, age32, calc32 = load_data_from_folder(LumB_path2,3,2,ns)
X_33, y_33, age33, calc33 = load_data_from_folder(LumB_path3,3,3,ns)
X_41, y_41, age41, calc41 = load_data_from_folder(TN_path1,4,1,ns)
X_42, y_42, age42, calc42 = load_data_from_folder(TN_path2,4,2,ns)
X_43, y_43, age43, calc43 = load_data_from_folder(TN_path3,4,3,ns)

100%|██████████| 52/52 [00:00<00:00, 326.69it/s]


In [8]:
# Combine all data into single arrays
num_classes = 5 
X = np.concatenate([X_0, X_11, X_12, X_13, X_21, X_22, X_23, X_31, X_32, X_33, X_41, X_42, X_43], axis=0)
y = np.concatenate([y_0, y_11, y_12, y_13, y_21, y_22, y_23, y_31, y_32, y_33, y_41, y_42, y_43], axis=0)
age = np.concatenate([age0, age11, age12, age13, age21, age22, age23, age31, age32, age33, age41, age42, age43], axis=0)
calc = np.concatenate([calc0, calc11, calc12, calc13, calc21, calc22, calc23, calc31, calc32, calc33, calc41, calc42, calc43], axis=0)
X_train_metadata = np.column_stack((age, calc))

In [9]:
#  First split: 72% training, 28% temporary (for validation and test)
X_train_images, X_temp, y_train, y_temp, X_train_metadata, metadata_temp = train_test_split(
    X, y, X_train_metadata, train_size=0.72, random_state=42, stratify=y  
)

# Second split: 18% validation, 10% test (from the remaining 28%)
X_val_images, X_test_images, y_val, y_test, X_val_metadata, X_test_metadata = train_test_split(
    X_temp, y_temp, metadata_temp, train_size=0.6428, random_state=42, stratify=y_temp  
)

print(f"Training set size: {len(X_train_images)}")
print(f"Validation set size: {len(X_val_images)}")
print(f"Test set size: {len(X_test_images)}")

y_train = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val = tf.keras.utils.to_categorical(y_val, num_classes=num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

Training set size: 2920
Validation set size: 730
Test set size: 406


In [ ]:
# Define the folder where you want to save the files
output_folder = "split_data2"
os.makedirs(output_folder, exist_ok=True) 

# Save the training, validation, and test datasets
np.save(os.path.join(output_folder, "X_train_images.npy"), X_train_images)
np.save(os.path.join(output_folder, "X_test_images.npy"), X_test_images)
np.save(os.path.join(output_folder, "X_train_metadata.npy"), X_train_metadata)
np.save(os.path.join(output_folder, "X_test_metadata.npy"), X_test_metadata)
np.save(os.path.join(output_folder, "y_train.npy"), y_train)
np.save(os.path.join(output_folder, "y_test.npy"), y_test)

# Save the validation datasets
np.save(os.path.join(output_folder, "X_val_images.npy"), X_val_images)
np.save(os.path.join(output_folder, "X_val_metadata.npy"), X_val_metadata)
np.save(os.path.join(output_folder, "y_val.npy"), y_val)

print(f"All datasets have been saved in the folder: {output_folder}")

All datasets have been saved in the folder: split_data2
